In [5]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

/opt/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
df = pd.read_csv("data/raw/cafes_seed.csv")

df["tags"] = df["tags"].fillna("").str.replace(",", " ")
df["category"] = df["category"].fillna("")
df["district"] = df["district"].fillna("")

df["search_text"] = (
    df["name"] + " " +
    df["district"] + " " +
    df["category"] + " " +
    df["tags"]
)

df.head()

,name,district,category,tags,search_text
0,분카샤 을지로점,을지로,dessert_cafe,dessert aesthetic photo,분카샤 을지로점 을지로 dessert_cafe dessert aesthetic photo
1,스쿠퍼젤라또,서촌,gelato,gelato dessert,스쿠퍼젤라또 서촌 gelato gelato dessert
2,카페폴리,합정,cafe,coffee minimal,카페폴리 합정 cafe coffee minimal
3,파크사이드,연남,cafe,rooftop wine date_spot,파크사이드 연남 cafe rooftop wine date_spot
4,오디티,서교,cafe,minimal quiet,오디티 서교 cafe minimal quiet


In [11]:
# Load the sentence transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14803.09it/s]


In [12]:
# Encode the search text to get embeddings
embeddings = model.encode(
    df["search_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True
)

In [13]:
# Build a FAISS index for efficient similarity search
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

In [14]:
# Recommendation function
# This function takes a user query, encodes it, and retrieves the most similar cafes from the dataset.
# It also allows optional filtering by district.
# Parameters:
# - query: The user's search query (string).
# - top_k: The number of top recommendations to return (default is 5).
# - district: Optional filter to restrict recommendations to a specific district (string).
# Returns: A DataFrame containing the recommended cafes with their details and similarity scores.
# Example usage:
# recommendations = semantic_recommend("cozy cafe with good coffee in Gangnam", top_k=5, district="Gangnam")
# This will return the top 5 cafes in the Gangnam district that are semantically similar to the query.
# Note: The function uses cosine similarity (via FAISS) to find the most relevant cafes based on the encoded search text.
def semantic_recommend(query, top_k=5, district=None):

    results_df = df.copy()

    if district:
        mask = results_df["district"].str.contains(district, case=False, na=False)
        candidate_indices = results_df[mask].index.tolist()
    else:
        candidate_indices = results_df.index.tolist()

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, len(df))

    recommendations = []

    for score, idx in zip(scores[0], indices[0]):
        if idx in candidate_indices:
            recommendations.append({
                "name": df.iloc[idx]["name"],
                "district": df.iloc[idx]["district"],
                "category": df.iloc[idx]["category"],
                "tags": df.iloc[idx]["tags"],
                "score": float(score)
            })

        if len(recommendations) >= top_k:
            break

    return pd.DataFrame(recommendations)

In [15]:
# Test the recommendation function with a sample query
semantic_recommend("quiet cafe in seongsu")

,name,district,category,tags,score
0,오디티,서교,cafe,minimal quiet,0.746470
1,카페 다우드,광장,cafe,quiet work_friendly,0.677412
2,정월,역삼,cafe,korean_style quiet,0.672597
3,신이도가,서교,cafe,hanok dessert quiet,0.645406
4,스탠딩밀포유,용산,cafe,vintage quiet,0.627807


In [16]:
semantic_recommend("good coffee for working")

,name,district,category,tags,score
0,피어커피 성수,성수,specialty_coffee,coffee work_friendly,0.714693
1,블루보틀 역삼,역삼,specialty_coffee,coffee work_friendly,0.698104
2,와이덴 성수,성수,cafe,dessert coffee,0.538626
3,로우커피스탠드,성수,specialty_coffee,coffee minimal,0.536245
4,퀸즈베이커리 옥수점,옥수,bakery,bakery coffee,0.530921


In [17]:
semantic_recommend("date cafe in hongdae")

,name,district,category,tags,score
0,그레이랩,합정,cafe,dessert date_spot,0.655577
1,파크사이드,연남,cafe,rooftop wine date_spot,0.559840
2,팬케익오리지널스토리 한남점,한남,brunch_cafe,brunch pancakes date_spot,0.542633
3,퍼프아웃,성수,cafe,aesthetic modern,0.484210
4,썸이프,대흥,cafe,dessert cozy,0.481019


In [19]:
# Test the recommendation function with a sample query and district filter
semantic_recommend("good coffee for working", district="성수")

,name,district,category,tags,score
0,피어커피 성수,성수,specialty_coffee,coffee work_friendly,0.714693
1,와이덴 성수,성수,cafe,dessert coffee,0.538626
2,로우커피스탠드,성수,specialty_coffee,coffee minimal,0.536245
3,챔프커피 성수,성수,specialty_coffee,coffee local_favorite,0.523931
4,맥코이 성수점,성수,specialty_coffee,coffee vintage,0.492010


In [20]:
# Save the processed data and FAISS index for later use
np.save("data/processed/cafe_embeddings.npy", embeddings)
df.to_csv("data/processed/cafes_with_search_text.csv", index=False)
faiss.write_index(index, "data/processed/cafe_faiss.index")